# YOLO26n 81종 학습: COCO 80종 + Hugging Face 드론

이 노트북은 **직접 실행해야 하는 학습용 도구**입니다. 완성된 모델이 아닙니다. GPU 필요, 데이터는 압축 파일만 약 30GB, 해제 후 수십 GB가 추가로 필요하며 학습은 며칠 걸릴 수 있습니다. Colab 무료 자원으로 완료되지 않을 수 있습니다. Seraphim 데이터 라이선스(CC BY 4.0)와 출처를 확인하세요.

**런타임 → 런타임 유형 변경 → GPU**로 설정한 뒤 위에서 아래로 실행하세요.


## 1. GPU/저장공간 검사 및 GitHub 코드 설치


In [ ]:
import subprocess, shutil, torch
from pathlib import Path
def run_logged(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout: print(line, end='', flush=True)
        if process.wait(): raise subprocess.CalledProcessError(process.returncode, command)
print('Free GiB:', round(shutil.disk_usage('/content').free/1024**3, 1))
if not torch.cuda.is_available(): raise RuntimeError('Colab에서 GPU 런타임을 선택해주세요')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['pip','-q','install','ultralytics>=8.4.142,<9','huggingface_hub>=0.28','pyyaml>=6'],check=True)
repo=Path('/content/yolo-ios-app')
if not repo.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/whgnltkd123-ai/yolo-ios-app.git',str(repo)],check=True)
training_ref='1e9724c9af237d2b119ce395977478101e147c1a'
run_logged(['git','-C',str(repo),'fetch','origin',training_ref])
run_logged(['git','-C',str(repo),'checkout','--detach',training_ref])


## 2. Hugging Face Seraphim 드론 이미지·YOLO 라벨 다운로드

약 9GB 압축+해제 공간이 필요합니다. 빈 GPU런타임과 저장공간을 확보하세요.


In [ ]:
from pathlib import Path
seraphim=Path('/content/seraphim')
run_logged(['python','-u',str(repo/'tools/download_hf_seraphim.py'),'--output',str(seraphim)])


## 3. 공식 COCO 80종 데이터 다운로드

약 20GB 이상 내려받습니다. COCO 전체가 있어야 기존 80종 보존을 실제로 시도할 수 있습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
training_project=Path('/content/drive/MyDrive/YOLO26-81-training')
training_project.mkdir(parents=True, exist_ok=True)
import ultralytics, yaml
from ultralytics.utils import ASSETS_URL
from ultralytics.utils.downloads import safe_download
coco=Path('/content/datasets/coco')
coco.mkdir(parents=True, exist_ok=True)
print('Free GiB before COCO:', round(shutil.disk_usage('/content').free/1024**3, 1))
if shutil.disk_usage('/content').free < 45*1024**3: raise RuntimeError('COCO download needs at least 45 GiB free')
# The merge script expects detection boxes, not default COCO segmentation polygons.
safe_download(ASSETS_URL+'/coco2017labels.zip', dir=coco.parent, exist_ok=True, delete=True)
for split, expected in [('train2017',118287),('val2017',5000)]:
    if len(list((coco/'images'/split).glob('*.jpg'))) != expected:
        safe_download('http://images.cocodataset.org/zips/'+split+'.zip', dir=coco/'images', exist_ok=True, delete=True)
    assert len(list((coco/'images'/split).glob('*.jpg'))) == expected
    assert (coco/'images'/split).is_dir() and (coco/'labels'/split).is_dir()
coco_eval=coco/'coco80-eval.yaml'
coco_names=yaml.safe_load((Path(ultralytics.__file__).parent/'cfg/datasets/coco.yaml').read_text())['names']
coco_eval.write_text(yaml.safe_dump({'path':str(coco),'train':'images/train2017','val':'images/val2017','names':coco_names}))
print('COCO dataset:', coco)


## 4. COCO 0~79 + drone 80으로 81종 통합


In [ ]:
merged=Path('/content/yolo26_81_ready')
run_logged(['python','-u',str(repo/'tools/prepare_yolo26_81_data.py'),
 '--coco-root',str(coco),'--drone-root',str(seraphim),
 '--drone-yaml',str(seraphim/'data.yaml'),'--drone-valid-split','test',
 '--output',str(merged)])
print('Combined dataset:', merged/'data.yaml')


## 5. 학습 시작

전체 약 20만 장을 60회 반복하므로 매우 오래 걸릴 수 있고 무료 Colab의 런타임이 중단될 수 있습니다. 별도 GPU 서버가 더 적합할 수 있습니다.


In [ ]:
last=training_project/'coco80-plus-drone/weights/last.pt'
command=['python','-u',str(repo/'tools/train_yolo26_81.py'),
 '--data',str(merged/'data.yaml'),'--epochs','60','--imgsz','640','--batch','8',
 '--device','0','--project',str(training_project),'--coco-val-data',str(coco_eval)]
if last.is_file(): command += ['--resume',str(last)]
run_logged(command)


## 6. 학습 완료 후 Google Drive로 저장

다음으로 모델 파일을 본인이 권한을 가진 HTTPS 저장소에 업로드하고 SHA256을 Codemagic `YOLO26 81 Classes` 워크플로에 입력해야 iPhone IPA가 빌드됩니다.


In [ ]:
from google.colab import drive
import hashlib
drive.mount('/content/drive')
best=training_project/'coco80-plus-drone/weights/best.pt'
if not best.is_file(): raise FileNotFoundError('학습 완료된 best.pt가 없습니다')
dest=Path('/content/drive/MyDrive/YOLO26-Drone-81-best.pt')
shutil.copy2(best,dest)
print('Model:',dest)
print('SHA256:',hashlib.sha256(dest.read_bytes()).hexdigest())
print('iPhone에서 실제 오탐/미탐과 FPS는 별도로 검증해야 합니다')
